## TrustyAI

<img src="https://trustyai.org/docs/main/_images/trustyai-logo-wide.svg" style="width:30%;">

Quelle: [TrustyAI](https://trustyai.org)
---

TrustyAI ist eine Open-Source-Komponente für erklärbare und überprüfbare KI-Anwendungen. 

Sie unterstützt unter anderem Modell-Erklärbarkeit, Fairness- und Drift-Analysen sowie Guardrails für generative KI.

In Kubernetes-Umgebungen kann TrustyAI beispielsweise vor einen ~~KServe~~ Ollama-Dienst geschaltet werden, um Prompts und Antworten anhand definierter Regeln zu prüfen.

---

Als erste Erstellen wir ein Zertifikat für TrustyAI


In [ ]:
%%bash
NAMESPACE=opendatahub
SERVICE=trustyai-service-operator-webhook-service

openssl req -x509 -newkey rsa:4096 \
  -keyout /tmp/webhook-server.key \
  -out /tmp/webhook-server.crt \
  -days 365 \
  -nodes \
  -subj "/CN=${SERVICE}.${NAMESPACE}.svc" \
  -addext "subjectAltName=DNS:${SERVICE},DNS:${SERVICE}.${NAMESPACE},DNS:${SERVICE}.${NAMESPACE}.svc,DNS:${SERVICE}.${NAMESPACE}.svc.cluster.local"

kubectl create secret tls webhook-server-cert \
  --cert=/tmp/webhook-server.crt \
  --key=/tmp/webhook-server.key \
  -n ${NAMESPACE}

**Kustomize**

Die Installation Verwendet kustomize, ein Kubernetes-Werkzeug, um YAML-Manifeste deklarativ anzupassen, ohne die Originaldateien direkt zu verändern. 

Wodurch dieselbe Basis-Konfiguration für verschiedene Umgebungen wie Test, Schulung und Produktion wiederverwendet werden kann. 

In [ ]:
%%bash
curl -sSLo /tmp/kustomize.tar.gz https://github.com/kubernetes-sigs/kustomize/releases/download/kustomize%2Fv5.7.1/kustomize_v5.7.1_linux_amd64.tar.gz

tar -xzf /tmp/kustomize.tar.gz -C /tmp
sudo install -m 0755 /tmp/kustomize /usr/local/bin/kustomize

kustomize version

In [ ]:
%%bash
rm -rf trustyai-service-operator
git clone https://github.com/trustyai-explainability/trustyai-service-operator.git
cd trustyai-service-operator

OPERATOR_NAMESPACE=opendatahub
make manifest-gen NAMESPACE=$OPERATOR_NAMESPACE KUSTOMIZE=kustomize

**Installation**

Nach der Aufbereitung erfolgt die eigentliche Installation von TrustyAI

In [ ]:
%%bash
cd trustyai-service-operator
kubectl create ns opendatahub
kubectl apply -n opendatahub -f release/trustyai_bundle.yaml

**Überprüfen**

Nach der Installation sollten Custom Resource Definition und ein Operator ersichtlich sein


In [ ]:
%%bash
kubectl get crd | grep trustyai
kubectl get pods -n opendatahub
kubectl get deployments -n opendatahub

---

### TrustyAI vorschalten

**TrustyAI vorschalten** bedeutet, dass Anfragen nicht direkt an das KI-Modell geschickt werden, sondern zuerst durch eine zusätzliche Prüfschicht laufen. Diese Schicht kann Prompts und Modellantworten analysieren, bevor sie an das Modell weitergeleitet beziehungsweise an den Benutzer zurückgegeben werden.

In einer KServe-Umgebung sieht das vereinfacht so aus:

```text
Direkter Zugriff:
Client → KServe → Modell

Geprüfter Zugriff:
Client → TrustyAI Guardrails → KServe → Modell
```

KServe bleibt dabei für das Bereitstellen des Modells zuständig. TrustyAI übernimmt die Rolle eines vorgeschalteten Kontrollpunkts. Dort können Regeln definiert werden, zum Beispiel um personenbezogene Daten, unerwünschte Inhalte oder bestimmte Muster in Eingaben und Ausgaben zu erkennen.


In [ ]:
%%bash
NAMESPACE=kserve-test
NAME=qwen-guardrails

openssl req -x509 -newkey rsa:4096 \
  -keyout /tmp/${NAME}.key \
  -out /tmp/${NAME}.crt \
  -days 365 \
  -nodes \
  -subj "/CN=127.0.0.1" \
  -addext "subjectAltName=IP:127.0.0.1,DNS:${NAME},DNS:${NAME}.${NAMESPACE},DNS:${NAME}.${NAMESPACE}.svc,DNS:${NAME}.${NAMESPACE}.svc.cluster.local,DNS:qwen-guardrails-service,DNS:qwen-guardrails-service.${NAMESPACE},DNS:qwen-guardrails-service.${NAMESPACE}.svc,DNS:qwen-guardrails-service.${NAMESPACE}.svc.cluster.local"

kubectl create secret tls ${NAME}-tls \
  --cert=/tmp/${NAME}.crt \
  --key=/tmp/${NAME}.key \
  -n ${NAMESPACE} \
  --dry-run=client -o yaml | kubectl apply -f -

**GuardrailsOrchestrator**

Der GuardrailsOrchestrator wird als Custom Resource erstellt. 

TrustyAI läuft als zweiter Einstiegspunkt, der KServe Port bleibt offen.

In [ ]:
%%bash
kubectl apply -f - <<EOF
apiVersion: v1
kind: ConfigMap
metadata:
  name: qwen-guardrails-config
  namespace: kserve-test
data:
  config.yaml: |
    chat_generation:
      service:
        hostname: ollama.ollama.svc.cluster.local
        port: 11434
    chunkers:
      my_chunker:
        type: all
        # WICHTIG: Das Pflichtfeld 'service' wird mit einer Dummy-Adresse befüllt
        service:
          hostname: "127.0.0.1"
          port: 8080
    detectors:
      regex:
        type: text_contents
        service:
          hostname: "127.0.0.1"
          port: 8080
        chunker_id: my_chunker
        default_threshold: 0.5
        patterns:
          - name: email
            regex: '[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}'
---
apiVersion: v1
kind: ConfigMap
metadata:
  name: qwen-guardrails-gateway-config
  namespace: kserve-test
data:
  config.yaml: |
    orchestrator:
      host: "qwen-guardrails-service.kserve-test.svc.cluster.local"
      port: 8032
      tls:
        cert_path: /etc/tls/private/tls.crt
    detectors:
      - name: regex
        server: regex
        input: true
        output: true
    routes:
      - name: pii
        detectors:
          - regex
---
apiVersion: trustyai.opendatahub.io/v1alpha1
kind: GuardrailsOrchestrator
metadata:
  name: qwen-guardrails
  namespace: kserve-test
spec:
  orchestratorConfig: qwen-guardrails-config
  enableBuiltInDetectors: true
  enableGuardrailsGateway: true
  guardrailsGatewayConfig: qwen-guardrails-gateway-config
  replicas: 1
  env:
    - name: SSL_CERT_FILE
      value: /etc/tls/private/tls.crt

EOF


In [ ]:
%%bash
# Reset Container
kubectl delete pod -l app=qwen-guardrails -n kserve-test

In [ ]:
%%bash
# HACK weil TrustyAI nur für OpenShift supported ist
kubectl patch deployment qwen-guardrails -n kserve-test --type='json' -p='[
  {
    "op": "add",
    "path": "/spec/template/spec/containers/0/securityContext",
    "value": {
      "runAsNonRoot": true,
      "runAsUser": 1001,
      "runAsGroup": 1001,
      "allowPrivilegeEscalation": false,
      "capabilities": {
        "drop": ["ALL"]
      }
    }
  }
]'

**Überprüfen**

Neben kserve läuft jetzt ein zweiter Pod mit den GuardRails

In [ ]:
%%bash
kubectl get guardrailsorchestrator -n kserve-test
kubectl get pods,services -n kserve-test

---

### Testen

Der Test erfolgt mittels dem OpenAI API.

In [ ]:
%%bash
kubectl patch service qwen-guardrails-service -n kserve-test --type='merge' -p '{ "spec": { "type": "LoadBalancer" } }'
PORT=$(kubectl get svc qwen-guardrails-service -n kserve-test -o jsonpath='{.spec.ports[?(@.port==8090)].nodePort}{"\n"}')

source ~/data/env.py
cat <<EOF | tee ~/data/env-kserve-nvidia.py
OPENAI_API_KEY="kserve-nvidia"
HF_TOKEN=""
AI_KUBECONFIG="$AI_KUBECONFIG"
AI_MODEL="llama3.1:8b-instruct-q4_K_M"
AI_NAME=""
AI_IP="${AI_IP}"
AI_BASE_URL="http://${AI_IP}:$PORT/pii/v1"
EOF

Zuerst eine einfache Abfrage ohne problematische Inhalte

In [ ]:
%run ~/data/env-kserve-nvidia.py
from openai import OpenAI

client = OpenAI(
    base_url=AI_BASE_URL,
    api_key="dummy",
)

response = client.chat.completions.create(
    model=AI_MODEL,
    messages=[
        {"role": "user", "content": "Erkläre Kubernetes kurz."}
    ],
    max_tokens=80,
    temperature=0.2,
)

print(response.choices[0].message.content)

Dann eine Abfrage mit Mailadresse, welche zurückgewiesen werden sollte

In [ ]:
%run ~/data/env-kserve-nvidia.py
from openai import OpenAI

client = OpenAI(
    base_url=AI_BASE_URL,
    api_key="dummy",
)

response = client.chat.completions.create(
    model=AI_MODEL,
    messages=[
        {"role": "user", "content": "Meine Mail ist test@example.com. Erkläre Kubernetes kurz."}
    ],
    max_tokens=80,
    temperature=0.2,
)

print(response.choices[0].message.content)